In [1]:
import os
import pandas as pd
import numpy as np
import argparse
import datetime
from tqdm import tqdm
from openai import OpenAI
from Levenshtein import distance


# 1) Input training and test data
For each antibody characterization scenario, the full dataset was randomly divided into the training set and test set. Due to cost considerations, we set each test set to contain 100 instances. The test sets were set balanced with equal number of instances per target class (Supplementary Table 3). All few-shot demonstrations were selected exclusively from the training set to ensure no data leakage in the prompts. 

In [2]:
train_df = pd.read_csv('../data/human_rhesus_no_align_train_set')
train_df

,seq,label_org,label
0,QVRLLESGPGLVTPSGTLSLPCDVSGHPISNSWWSWVRQPPGKGLE...,human,1
1,QLQLQESGPGLGKPSETLSLTCAVSGGSIRSNYWSWRRQSPGKGLE...,rhesus,0
2,QVQLQESGPGLVKPSETLSLTCAVSGYSISTGYGWSWIRQPPGKGL...,rhesus,0
3,EVQLVQSGAEVKKPGESLKISCKGSGYSFTSYWIGWVRQMPGKGLE...,human,1
4,QPQLVGSGGGVVQPGRSLTLSCAAAGFSFSNYAMEWVRQIPGKGPE...,human,1
...,...,...,...
19895,EVQLLESGGGLVQPGGSLRLSCAASGFTFNFYAMSWFRQAPGKGLE...,human,1
19896,QVQLQQWGAGLLKPSETLSLTCAVYGGSFSSYYFSWIRQPPGKGLE...,human,1
19897,HVQLQESGPGLVKPSETLSLTCTVSGDFPSDDHCTWIRQAPGKGLE...,human,1
19898,QVQLQESGPGLVKPSETLSLTCAVSGGSISDDYYWSWIRQPPGKGL...,rhesus,0


In [3]:
test_df = pd.read_csv('../data/human_rhesus_no_align_test_set')
test_df

,seq,label_org,label
0,QVQLQQWGEGLVKPSETLSLTCAVYGGSVSGYWWGWIRPPPGKGLE...,rhesus,0
1,QVQLQESGPGLVKPSETLSLTCTVSGYSIKTGYGWNWIRQPPVKGM...,rhesus,0
2,EVQLVQSGAEVKKPGESLKISCKGSGYSFTSYWIGWVRQMPGKGLE...,human,1
3,QVQLVQSGAEVKKPGASVKVSCKASGYTFTNYGVSWVRQAPGQGLE...,human,1
4,QVQLQESGPGVVKPSETLSLTCAVSGGSISSGYDWSWIRQPPGKGL...,rhesus,0
...,...,...,...
95,EEQVVESGGGLVQPGGSLRLSCAASGFTFNMYELSWVRQAPGKRLE...,human,1
96,EVQLVQSGAEVKKPGASVKISCKASGYTFTDHYLNWVRQAPGKGLE...,rhesus,0
97,EEQLLESGGGLVQPGDSLRLSCAASGFRFSNYWMNWVRQAPGKGLV...,human,1
98,QVQLQESGPGLVKPSETLSLSCSVSGGSISSFFWNWIRQPPGRGLE...,human,1


In [4]:
any([seq in test_df['seq'].to_list() for seq in train_df['seq']])

False

In [5]:
any([seq in train_df['seq'].to_list() for seq in train_df['seq']])

True

# 2) Demonstration selection
For the random strategy, i.e., Random (Balanced), an equal number of instances from each target class were randomly selected exclusively from the training set.

For the amino acid letter code-based representation in the similarity-based strategies, including both the Similar and the Similar (Balanced) strategies, sequence similarity was assessed using LD between each training instance and each test instance. For the Similar (Balanced) strategy, an equal number of similar instances were selected from each target class. In k-shot learning, we chose the k most similar k training instances (those with the smallest LD) for each test instance.


In [6]:
def leven_distmat(train_set, test_instance, sample_size):
    dist = np.asarray([distance(test_instance, train_str) for train_str in train_set[selected_chain]])
    dist_idx = dist.argsort()[:sample_size][::-1]
    return dist_idx

def random_sample_examples(input_train, sample_size):
    pos = input_train[input_train["label"] == 1].sample(n=int(sample_size/2))
    neg = input_train[input_train["label"] == 0].sample(n=int(sample_size/2))
    bcr = pos[selected_chain].tolist() + neg[selected_chain].tolist()
    class_label = pos["label"].tolist() + neg["label"].tolist()
    #convert 1 to "Yes" and 0 to "No"" in class_label
    # class_label = ["Yes" if i == 1 else "No" for i in class_label]
    bcr_examples = list(zip(bcr, class_label))
    return bcr_examples

def similar_sample_examples(input_train_df, test_instance, sample_size, mode=''):
    dist_idx = leven_distmat(input_train_df, test_instance, sample_size)
    if mode == 'reverse':
        rev_idx = np.flip(dist_idx)
        similar_fewshot = input_train_df.iloc[rev_idx,:]
    else:
        similar_fewshot = input_train_df.iloc[dist_idx,:]
    bcr = similar_fewshot[selected_chain].tolist()
    class_label = similar_fewshot['label'].tolist()
    # class_label = ["Yes" if i == 1 else "No" for i in class_label]
    bcr_examples = list(zip(bcr, class_label))
    return bcr_examples

def similar_balanced_sample_examples(input_train_df, test_instance, sample_size, mode):
    pos_train = input_train_df[input_train_df['label']==1]
    neg_train = input_train_df[input_train_df['label']==0]
    pos_idx = leven_distmat(pos_train, test_instance, sample_size//2)
    neg_idx = leven_distmat(neg_train, test_instance, sample_size//2)
    pos_fewshot = pos_train.iloc[pos_idx]
    neg_fewshot = neg_train.iloc[neg_idx]
    if mode == 'pos_neg':
        similar_fewshot = pd.concat([pos_fewshot, neg_fewshot])
    elif mode == 'pos':
        similar_fewshot = pos_fewshot
    elif mode == 'neg_pos':
        similar_fewshot = pd.concat([neg_fewshot, pos_fewshot])
    elif mode == 'random':
        similar_fewshot = pd.concat([pos_fewshot, neg_fewshot])
        similar_fewshot = similar_fewshot.sample(frac=1, random_state=random_state)
    else:
        print(f'Your input mode is invalid')
    bcr = similar_fewshot[selected_chain].tolist()
    class_label = similar_fewshot['label'].tolist()
    # class_label = ["Yes" if i == 1 else "No" for i in class_label]
    bcr_examples = list(zip(bcr, class_label))
    return bcr_examples

In [7]:
random_state=42
selected_chain = 'seq'
sample_method = 'similar_balanced_random'
sample_num = 16

In [8]:
print(f'Processing: {sample_method} Few-shot: {sample_num}')
selected_test_df = test_df.loc[:,[selected_chain,'label']]
test_example = selected_test_df.to_records(index=False)
demo_list = []
for test in tqdm(test_example):
    if sample_method == 'random':
        bcr_examples = random_sample_examples(train_df,sample_num)
    elif sample_method == 'similar':
        bcr_examples = similar_sample_examples(train_df, test[0], sample_num)
    elif sample_method == 'similar_reverse':
        bcr_examples = similar_sample_examples(train_df, test[0], sample_num, mode='reverse')
    elif sample_method == 'similar_balanced_posneg':
        bcr_examples = similar_balanced_sample_examples(train_df, test[0], sample_num, mode='pos_neg')
    elif sample_method == 'similar_balanced_pos':
        bcr_examples = similar_balanced_sample_examples(train_df, test[0], sample_num, mode='pos')
    elif sample_method == 'similar_balanced_negpos':
        bcr_examples = similar_balanced_sample_examples(train_df, test[0], sample_num, mode='neg_pos')
    elif sample_method == 'similar_balanced_random':
        bcr_examples = similar_balanced_sample_examples(train_df, test[0], sample_num, mode='random')
    else:
        print('The selected method is unavailable.')
    demo_list.append(bcr_examples)


  3%|▎         | 3/100 [00:00<00:04, 20.40it/s]

Processing: similar_balanced_random Few-shot: 16


100%|██████████| 100/100 [00:04<00:00, 21.24it/s]


# 3) Prompt generation
A prompt refers to a set of instructions and context provided as input to guide the language model’s response generation. In most APIs, prompts can be provided to different roles, including “system”, “user”, and “assistant” roles. The “system” role, included at the start of the prompt, provides context, instructions, or other information to prime the model for the specific use case. The “user” role contains the actual user queries. The “assistant” role contains the model’s responses to the user queries, based on the context provided in the “system” role and “user” role. These roles work together, with the “system” message setting the stage, the “user” message providing the actual user queries, and the “assistant” message containing the model’s response to the user queries considering the overall context.

All prompts in this work were input into the “user” role, unless otherwise specified. The prompts were designed following previous publications and prompt engineering guides from OpenAI and Anthropic. 

In [9]:
def create_prompt(test_example, fewshot_examples, example_tag=False):
    if example_tag == True:
        prompt = f"You are an AI model specializing in immunology. Your task is to predict the humanness of antibody using only the amino acid sequences provided. Recent research has suggested that the humanness of antibody can be predicted with antibody sequence alone, without any structural information.\nPlease strictly follow the format, no other information can be provided. Given the amino acid sequences of a antibody, predict the humanness of antibody based on its sequence, by analyzing whether it can Human(1) or {organism}(0). Consider factors such as physiochemical property, PSSM and amino acid frequency to assess the humanness of antibody.\nPlease answer with *only* 1 or 0 only for the last example. A few examples are provided in the beginning.\n<example>\n"
        for example in fewshot_examples:
            prompt += f"Antibody: {example[0]}\nHumanness: {example[-1]}\n"
        prompt += "</example>\nNow, predict the humanness of antibody for the following antibody sequence based on the criteria above:\n"
    else:
        prompt = f"You are an AI model specializing in immunology. Your task is to predict the humanness of antibody using only the amino acid sequences provided. Recent research has suggested that the humanness of antibody can be predicted with antibody sequence alone, without any structural information.\nPlease strictly follow the format, no other information can be provided. Given the amino acid sequences of a antibody, predict the humanness of antibody based on its sequence, by analyzing whether it can Human(1) or {organism}(0). Consider factors such as physiochemical property, PSSM and amino acid frequency to assess the humanness of antibody.\nPlease answer with *only* 1 or 0 only for the last example. A few examples are provided in the beginning.\n"
        for example in fewshot_examples:
            prompt += f"Antibody: {example[0]}\nHumanness: {example[-1]}\n"
        prompt += "\nNow, predict the humanness of antibody for the following antibody sequence based on the criteria above:\n"
    prompt += f"Antibody: {test_example}\nHumanness:\n"
    return prompt

In [10]:
organism = 'rhesus'

In [11]:
prompt_list = []
test_examples = selected_test_df.to_records(index=False)

for test, few_shot in tqdm(zip(test_examples, demo_list)):
    prompt_list.extend([create_prompt(test[0], few_shot, example_tag=False)])

100it [00:00, 71428.88it/s]


In [12]:
print(prompt_list[0])

You are an AI model specializing in immunology. Your task is to predict the humanness of antibody using only the amino acid sequences provided. Recent research has suggested that the humanness of antibody can be predicted with antibody sequence alone, without any structural information.
Please strictly follow the format, no other information can be provided. Given the amino acid sequences of a antibody, predict the humanness of antibody based on its sequence, by analyzing whether it can Human(1) or rhesus(0). Consider factors such as physiochemical property, PSSM and amino acid frequency to assess the humanness of antibody.
Please answer with *only* 1 or 0 only for the last example. A few examples are provided in the beginning.
Antibody: QVQLQQWGAGLLKPSETLSLTCAVSGGSFSGYYWSWIRQPPGKGLEWIGEINHSGSTNYNPSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYCARIQGPFDYWGQGTLVTVSS
Humanness: 1
Antibody: QVQLQQWGAGLLKPSETLSLTCAVYGGSFSGYYWSWIRQPPGKGLEWIGEINHSGSTNYNQSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYCARGVWNDETDYWGQGTLV

# 4) Input ICL prompts into LLM (i.e., GPT)
If an unexpected API interruption occurs (e.g., network interruption), the entire prediction process will restart from the beginning. All the LLMs used a temperature of 0.2, and all the other API parameters were set to default unless specified. For each test instance in each setting, we ran its prompt five times to evaluate the consistency of the results.

In [13]:
def generate_response_by_gpt(prompt, model_engine):
    api_key = os.environ['OPENAI_API_KEY'] #enter your openai api key here
    client = OpenAI(api_key=api_key)
    completion = client.chat.completions.create(
      model=model_engine,
      messages = [{"role": "user", "content": prompt}],
      temperature=tempt, n=5
    )
    message = completion.choices
    message = [i.message.content.strip() for i in message]
    session_id = completion.id
    return message, session_id

In [14]:
output_name = 'scenario1b_antibody_humanness'
predict_file = f'./{output_name}_pred'
log_file = f'./{output_name}.log'

In [15]:
test_list = [prompt.split('\n')[-3].split(' ')[-1] for prompt in prompt_list]
generated_response = []
model = 'gpt-4o-2024-08-06'
tempt = 0.2

In [16]:
for idx, prompt in tqdm(enumerate(prompt_list)):
    current_runs = idx+1
    test_df = pd.DataFrame(test_list[0:idx+1], columns=['input'])
    generated_p, session_id = generate_response_by_gpt(prompt, model)
    with open(log_file, "a") as file:
        now = datetime.datetime.now()
        date_time_str = now.strftime("%Y-%m-%d %H:%M:%S")
        file.write("=" * 30 + date_time_str + "| Session ID: " + session_id + "=" * 30 + "\n")
        file.write(prompt + "\n")
    print(generated_p)
    generated_response.append(generated_p)
    if current_runs % 10 == 0:
        result_df = pd.DataFrame(generated_response, columns=['pred_1', 'pred_2', 'pred_3', 'pred_4', 'pred_5'])
        final_df = pd.concat([test_df, result_df], axis=1)
        print(final_df)
        final_df.to_csv(f'{predict_file}.csv', index=False)

1it [00:02,  2.07s/it]

['0', '0', '0', '0', '0']


2it [00:03,  1.77s/it]

['0', '0', '0', '0', '0']


3it [00:04,  1.65s/it]

...

['1', '1', '1', '1', '1']


99it [01:57,  1.12s/it]

['1', '1', '1', '1', '1']


100it [01:59,  1.20s/it]

['1', '0', '0', '0', '0']
                                                input pred_1 pred_2 pred_3  \
0   QVQLQQWGEGLVKPSETLSLTCAVYGGSVSGYWWGWIRPPPGKGLE...      0      0      0   
1   QVQLQESGPGLVKPSETLSLTCTVSGYSIKTGYGWNWIRQPPVKGM...      0      0      0   
2   EVQLVQSGAEVKKPGESLKISCKGSGYSFTSYWIGWVRQMPGKGLE...      1      1      1   
3   QVQLVQSGAEVKKPGASVKVSCKASGYTFTNYGVSWVRQAPGQGLE...      1      1      1   
4   QVQLQESGPGVVKPSETLSLTCAVSGGSISSGYDWSWIRQPPGKGL...      0      0      0   
..                                                ...    ...    ...    ...   
95  EEQVVESGGGLVQPGGSLRLSCAASGFTFNMYELSWVRQAPGKRLE...      1      1      1   
96  EVQLVQSGAEVKKPGASVKISCKASGYTFTDHYLNWVRQAPGKGLE...      0      0      0   
97  EEQLLESGGGLVQPGDSLRLSCAASGFRFSNYWMNWVRQAPGKGLV...      1      1      1   
98  QVQLQESGPGLVKPSETLSLSCSVSGGSISSFFWNWIRQPPGRGLE...      1      1      1   
99  EVLLVESGGGVVQPGGYLRLSCAASGFNFTSSWMSWVRQAPGRGLE...      1      0      0   

   pred_4 pred_5  
0       0      0  

In [17]:
result_df = pd.DataFrame(generated_response, columns=['pred_1', 'pred_2', 'pred_3', 'pred_4', 'pred_5'])
final_df = pd.concat([test_df, result_df], axis=1)
final_df.to_csv(f'{predict_file}.csv', index=False)

## 5) Performance evaluation

To convert the responses into predicted labels, we used the following criteria: across all the scenarios, a predicted label of 1 was assigned to formatted responses of “1”, “1 \n”, “1\n” or “*1*”. For unformatted responses (Supplementary Table 4), specific phrases were extracted for conversion. In Scenario 1, a predicted label of 1 was assigned to responses containing “Humanness: 1” or “Answer: 1”. The remaining responses were classified as having a predicted label of 0 in all the scenarios.

After converting the responses into labels, accuracy was calculated as the evaluation metric by comparing the predictions with the true labels of the test set. Each experiment was repeated five times, and the accuracy was averaged across these five repetitions, with the final average accuracy being reported.

In [18]:
from sklearn.metrics import accuracy_score
from collections import Counter

In [19]:
test_df = pd.read_csv('../data/human_rhesus_no_align_test_set')
test_df

,seq,label_org,label
0,QVQLQQWGEGLVKPSETLSLTCAVYGGSVSGYWWGWIRPPPGKGLE...,rhesus,0
1,QVQLQESGPGLVKPSETLSLTCTVSGYSIKTGYGWNWIRQPPVKGM...,rhesus,0
2,EVQLVQSGAEVKKPGESLKISCKGSGYSFTSYWIGWVRQMPGKGLE...,human,1
3,QVQLVQSGAEVKKPGASVKVSCKASGYTFTNYGVSWVRQAPGQGLE...,human,1
4,QVQLQESGPGVVKPSETLSLTCAVSGGSISSGYDWSWIRQPPGKGL...,rhesus,0
...,...,...,...
95,EEQVVESGGGLVQPGGSLRLSCAASGFTFNMYELSWVRQAPGKRLE...,human,1
96,EVQLVQSGAEVKKPGASVKISCKASGYTFTDHYLNWVRQAPGKGLE...,rhesus,0
97,EEQLLESGGGLVQPGDSLRLSCAASGFRFSNYWMNWVRQAPGKGLV...,human,1
98,QVQLQESGPGLVKPSETLSLSCSVSGGSISSFFWNWIRQPPGRGLE...,human,1


In [20]:
def performance(final_df):
    input_df = pd.concat([final_df, test_df], axis=1)
    metrics = {}
    print(Counter(input_df['label']))
    for i in range(1,6):
        current_pred = f'pred_{i}'
        input_df[current_pred] = [1 if pred == 1 or pred == '1' or pred == '1 \n' or pred == '1\n' or pred == '*1*' or "Answer: 1" in str(pred) or 'Humanness: 1' in str(pred) else 0 for pred in input_df[current_pred]]
        score = accuracy_score(input_df['label'], input_df[current_pred])
        metrics[current_pred] = score         
    return metrics

In [21]:
output_performance = pd.DataFrame(performance(final_df), index=['acc']).T

Counter({0: 50, 1: 50})


In [22]:
output_performance

,acc
pred_1,0.94
pred_2,0.93
pred_3,0.93
pred_4,0.93
pred_5,0.93
